# Korean LLM QLoRA fine-tuning (Kaggle T4)

Qwen2.5 Instruct 모델을 한국어 instruction 데이터로 QLoRA 학습하고, 학습 전후를 같은 기준으로 측정해 결과 파일을 남깁니다.

**Kaggle 설정:** Settings → Accelerator **GPU T4 x2**, Internet **On**. 학습은 GPU 한 장만 씁니다.

**결과 파일 (`/kaggle/working/results/`)**
- `metrics.json`: held-out loss와 perplexity의 학습 전후 값, 학습 시간, GPU, 라이브러리 버전
- `loss_curve.png`, `log_history.json`: train loss와 eval loss 기록
- `samples_before_after.md`: 같은 질문에 대한 학습 전후 답변과 참조 답변
- `adapter/`: 학습된 LoRA adapter


In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"      # 4-bit 모델은 GPU 한 장에서 학습
os.environ["TOKENIZERS_PARALLELISM"] = "false"
!nvidia-smi

In [ ]:
!pip install -q -U transformers peft accelerate bitsandbytes datasets
import torch, transformers, peft, accelerate, bitsandbytes, datasets
print({m.__name__: m.__version__ for m in [torch, transformers, peft, accelerate, bitsandbytes, datasets]})
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# ===== 설정 =====
MODEL_ID   = "Qwen/Qwen2.5-1.5B-Instruct"   # 첫 실행. 잘 되면 "Qwen/Qwen2.5-3B-Instruct"로 바꿔서 다시
DATASET_ID = "beomi/KoAlpaca-v1.1a"         # 공개 한국어 instruction 데이터 (instruction, output)
N_TRAIN, N_EVAL, N_SAMPLES = 2000, 200, 20  # 학습, held-out 평가, 학습 전후 비교용 질문 수
MAX_LEN = 512
SEED = 42
OUT = "/kaggle/working/results"
os.makedirs(OUT, exist_ok=True)
SYSTEM = "당신은 친절하고 정확한 한국어 상담 어시스턴트입니다. 질문의 핵심부터 간결하게 답하세요."
LORA = dict(r=16, lora_alpha=32, lora_dropout=0.05,
            target_modules=["q_proj", "k_proj", "v_proj", "o_proj"])

In [ ]:
# ===== 데이터: 학습, held-out 평가, 비교용 질문을 겹치지 않게 나눔 =====
from datasets import load_dataset
raw = load_dataset(DATASET_ID, split="train").shuffle(seed=SEED)
raw = raw.filter(lambda x: bool(x["instruction"]) and bool(x["output"]) and len(x["output"]) > 20)
split = raw.train_test_split(test_size=N_EVAL + N_SAMPLES, seed=SEED)
train_raw  = split["train"].select(range(min(N_TRAIN, len(split["train"]))))
eval_raw   = split["test"].select(range(N_EVAL))
sample_raw = split["test"].select(range(N_EVAL, N_EVAL + N_SAMPLES))
print(len(train_raw), len(eval_raw), len(sample_raw))

In [ ]:
# ===== 토큰화: 답변 부분만 loss에 들어가도록 질문 부분은 -100으로 가림 =====
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(MODEL_ID)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

def to_messages(q, a=None):
    m = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": q}]
    if a is not None:
        m.append({"role": "assistant", "content": a})
    return m

def encode(ex):
    prompt_text = tok.apply_chat_template(to_messages(ex["instruction"]), add_generation_prompt=True, tokenize=False)
    full_text   = tok.apply_chat_template(to_messages(ex["instruction"], ex["output"]), tokenize=False)
    assert full_text.startswith(prompt_text)
    prompt_ids = tok(prompt_text, add_special_tokens=False)["input_ids"]
    full_ids   = tok(full_text, add_special_tokens=False)["input_ids"]
    labels = [-100] * len(prompt_ids) + full_ids[len(prompt_ids):]
    full_ids, labels = full_ids[:MAX_LEN], labels[:MAX_LEN]
    return {"input_ids": full_ids, "attention_mask": [1] * len(full_ids), "labels": labels}

def build(ds):
    out = ds.map(encode, remove_columns=ds.column_names)
    return out.filter(lambda x: any(l != -100 for l in x["labels"]))   # 답변이 잘려 사라진 예시는 제외

train_ds, eval_ds = build(train_raw), build(eval_raw)
print("train", len(train_ds), "eval", len(eval_ds))
ex = train_ds[0]
print("prompt tokens masked:", sum(l == -100 for l in ex["labels"]), "/ total:", len(ex["labels"]))

In [ ]:
# ===== 배치 묶기 =====
import torch
def collate(batch):
    n = max(len(b["input_ids"]) for b in batch)
    ids = torch.full((len(batch), n), tok.pad_token_id, dtype=torch.long)
    lab = torch.full((len(batch), n), -100, dtype=torch.long)
    att = torch.zeros((len(batch), n), dtype=torch.long)
    for i, b in enumerate(batch):
        k = len(b["input_ids"])
        ids[i, :k] = torch.tensor(b["input_ids"])
        lab[i, :k] = torch.tensor(b["labels"])
        att[i, :k] = 1
    return {"input_ids": ids, "attention_mask": att, "labels": lab}

In [ ]:
# ===== 4-bit 모델과 LoRA =====
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)   # T4는 bf16 미지원
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb, device_map={"": 0})
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
model = get_peft_model(model, LoraConfig(bias="none", task_type="CAUSAL_LM", **LORA))
for p in model.parameters():          # fp16 학습에서 LoRA 가중치는 fp32로 유지
    if p.requires_grad:
        p.data = p.data.float()
model.print_trainable_parameters()

In [ ]:
# ===== 답변 생성 (학습 전후 비교용, greedy로 고정) =====
@torch.no_grad()
def answer(q, max_new_tokens=256):
    model.eval()
    text = tok.apply_chat_template(to_messages(q), add_generation_prompt=True, tokenize=False)
    enc = tok(text, return_tensors="pt", add_special_tokens=False).to(model.device)
    out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                         repetition_penalty=1.1, pad_token_id=tok.pad_token_id, use_cache=True)
    return tok.decode(out[0, enc["input_ids"].shape[1]:], skip_special_tokens=True).strip()

questions = list(sample_raw["instruction"])
before = [answer(q) for q in questions]      # LoRA 초기값은 0이라 이 시점의 출력은 base 모델과 같음
print(before[0][:300])

In [ ]:
# ===== 학습: 학습 전과 후에 같은 held-out set으로 평가 =====
import time
from transformers import TrainingArguments, Trainer
steps_per_epoch = -(-len(train_ds) // (4 * 4))          # batch 4 x grad accum 4
warmup = max(1, int(0.03 * steps_per_epoch))             # warmup 3%
args = TrainingArguments(
    output_dir="/kaggle/working/ckpt",
    per_device_train_batch_size=4, per_device_eval_batch_size=4, gradient_accumulation_steps=4,
    num_train_epochs=1, learning_rate=2e-4, lr_scheduler_type="cosine", warmup_steps=warmup, weight_decay=0.01,
    logging_steps=10, eval_strategy="steps", eval_steps=25, save_strategy="no",
    fp16=True, optim="paged_adamw_8bit", gradient_checkpointing=True,
    report_to="none", seed=SEED, remove_unused_columns=False)
trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=eval_ds, data_collator=collate)
eval_before = trainer.evaluate()
print("eval before:", eval_before)
t0 = time.time()
trainer.train()
train_minutes = (time.time() - t0) / 60
eval_after = trainer.evaluate()
print("eval after:", eval_after, "| train minutes:", round(train_minutes, 1))

In [ ]:
# ===== 결과 저장 =====
import json, math
import matplotlib.pyplot as plt
hist = trainer.state.log_history
json.dump(hist, open(f"{OUT}/log_history.json", "w"), ensure_ascii=False, indent=1)

tr = [(h["step"], h["loss"]) for h in hist if "loss" in h]
ev = [(h["step"], h["eval_loss"]) for h in hist if "eval_loss" in h]
plt.figure(figsize=(7, 4))
if tr: plt.plot(*zip(*tr), label="train loss")
if ev: plt.plot(*zip(*ev), marker="o", label="held-out eval loss")
plt.axhline(eval_before["eval_loss"], linestyle="--", color="gray", label="eval loss before training")
plt.xlabel("step"); plt.ylabel("loss"); plt.title(MODEL_ID + " QLoRA"); plt.legend(); plt.tight_layout()
plt.savefig(f"{OUT}/loss_curve.png", dpi=150)

after = [answer(q) for q in questions]
with open(f"{OUT}/samples_before_after.md", "w") as f:
    for i, (q, ref, b, a) in enumerate(zip(questions, sample_raw["output"], before, after), 1):
        f.write(f"## {i}. {q}\n\n**Reference**\n\n{ref}\n\n**Before**\n\n{b}\n\n**After**\n\n{a}\n\n---\n\n")

metrics = {
    "model": MODEL_ID, "dataset": DATASET_ID,
    "n_train": len(train_ds), "n_eval_heldout": len(eval_ds), "n_samples": len(questions), "max_len": MAX_LEN,
    "lora": LORA, "quantization": "4-bit NF4, double quant, fp16 compute",
    "eval_loss_before": round(eval_before["eval_loss"], 4), "eval_loss_after": round(eval_after["eval_loss"], 4),
    "perplexity_before": round(math.exp(eval_before["eval_loss"]), 2), "perplexity_after": round(math.exp(eval_after["eval_loss"]), 2),
    "train_minutes": round(train_minutes, 1), "gpu": torch.cuda.get_device_name(0),
    "versions": {m.__name__: m.__version__ for m in [torch, transformers, peft, bitsandbytes, datasets]},
}
json.dump(metrics, open(f"{OUT}/metrics.json", "w"), ensure_ascii=False, indent=1)
model.save_pretrained(f"{OUT}/adapter"); tok.save_pretrained(f"{OUT}/adapter")
print(json.dumps(metrics, ensure_ascii=False, indent=1))

## 다음 단계
1. 오른쪽 **Output**에서 `results/` 폴더를 받아 `korean-llm-finetuning` 저장소의 `results/qlora_<모델>/`에 그대로 커밋합니다.
2. `samples_before_after.md`를 직접 읽고, 좋아진 점과 나빠진 점을 README에 솔직하게 적습니다. loss가 내려가도 답변이 나빠질 수 있습니다.
3. 1.5B가 잘 되면 `MODEL_ID`를 3B로 바꿔 다시 돌립니다. AI Hub 상담 데이터 승인이 나면 `DATASET_ID` 부분만 그 데이터로 바꿉니다.
